# 06. Phase C Report

## UDFPS PINN Platform — Phase C Results

### Executive Summary

| Metric | Target | Result | Status |
|--------|--------|--------|--------|
| BM boundary |U| | < 0.05 | TBD | - |
| ASM matching error | < 10% | TBD | - |
| Interior fringe std | > 0.1 | TBD | - |
| Design sensitivity | > 0.01 | TBD | - |
| Red flags | None | TBD | - |

*This notebook will be populated after GPU training completes.*

In [ ]:
import sys
from pathlib import Path

def _find_root():
    p = Path.cwd().resolve()
    for _ in range(10):
        if (p / 'pyproject.toml').exists(): return p
        p = p.parent
    raise FileNotFoundError('Cannot find project root')

PROJECT_ROOT = _find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch, json, yaml
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from backend.core.pinn_model import PurePINN
from backend.training.loss_functions import ASMIncidentLUT
from backend.training.red_flag_detector import detect_red_flags
from backend.physics.boundary_conditions import compute_is_bm
print('Ready')

---
## 1. Training Summary

In [ ]:
exp_dirs = sorted((PROJECT_ROOT / 'experiments').glob('*phase_c*'),
                  key=lambda p: p.stat().st_mtime, reverse=True)

if not exp_dirs:
    print('No experiments yet. Run scripts/train_phase_c.py first.')
else:
    EXP = exp_dirs[0]
    print(f'Experiment: {EXP.name}')
    
    if (EXP / 'config.yaml').exists():
        with open(EXP / 'config.yaml') as f:
            cfg = yaml.safe_load(f)
        print(f'Epochs: {cfg["training"]["epochs"]}')
        print(f'Model: hidden={cfg["model"]["hidden_dim"]}, layers={cfg["model"]["num_layers"]}')
        print(f'Device: {cfg["training"]["device"]}')
    
    if (EXP / 'loss_history.json').exists():
        with open(EXP / 'loss_history.json') as f:
            hist = json.load(f)
        
        fig, ax = plt.subplots(figsize=(12, 4))
        ax.semilogy(hist['total'], 'k-', lw=1, label='Total')
        ax.semilogy(hist['L_H'], 'b-', alpha=0.6, label='L_H')
        ax.semilogy(hist['L_phase'], 'r-', alpha=0.6, label='L_phase')
        ax.semilogy(hist['L_BC'], 'g-', alpha=0.6, label='L_BC')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss')
        ax.set_title('Training Loss History')
        ax.legend()
        ax.grid(True, alpha=0.2)
        plt.tight_layout()
        plt.show()
        
        print(f'Final: Total={hist["total"][-1]:.4e} H={hist["L_H"][-1]:.4e} '
              f'Phase={hist["L_phase"][-1]:.4e} BC={hist["L_BC"][-1]:.4e}')

---
## 2. Key Results

In [ ]:
ckpt_path = PROJECT_ROOT / 'checkpoints' / 'phase_c_final.pt'
if not ckpt_path.exists():
    ckpts = sorted((PROJECT_ROOT / 'checkpoints').glob('phase_c_*.pt'),
                   key=lambda p: p.stat().st_mtime, reverse=True)
    ckpt_path = ckpts[0] if ckpts else None

if ckpt_path:
    if exp_dirs and (exp_dirs[0] / 'config.yaml').exists():
        mcfg = cfg['model']
    else:
        mcfg = {'hidden_dim': 128, 'num_layers': 4, 'num_freqs': 48, 'omega_0': 30.0}
    
    model = PurePINN(**mcfg)
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    
    report = detect_red_flags(model, torch.device('cpu'))
    
    # Summary table
    checks = [
        ('BM boundary', report.bm1_mean_amp < 0.05 and report.bm2_mean_amp < 0.05,
         f'BM1={report.bm1_mean_amp:.4f}, BM2={report.bm2_mean_amp:.4f}'),
        ('Interior fringe', report.interior_cov > 0.05,
         f'CoV={report.interior_cov:.4f}'),
        ('Design sensitivity', report.design_sensitivity > 0.01,
         f'range={report.design_sensitivity:.4f}'),
        ('No red flags', not report.has_red_flag, report.summary()),
    ]
    
    print('='*60)
    print('PHASE C RESULTS')
    print('='*60)
    all_pass = True
    for name, passed, detail in checks:
        status = 'PASS' if passed else 'FAIL'
        icon = '+' if passed else 'X'
        print(f'  [{icon}] {name:25s} {status:6s}  {detail}')
        all_pass = all_pass and passed
    print('='*60)
    print(f'Overall: {"PHASE C COMPLETE" if all_pass else "NEEDS MORE WORK"}')
    if all_pass:
        print('Next: Phase D (FNO distillation + BoTorch inverse design)')
    print('='*60)
else:
    print('No checkpoint found.')

---
## 3. Phase 1 vs Phase C Comparison

In [ ]:
print('Phase 1 Results (baseline):')
print('  MTF@ridge: 99.78%')
print('  PSF skewness: 0.075')
print('  Hypervolume: 0.464')
print('  Design variables: 5D (AR 4 + delta_BM)')
print('  PINN domain: 30 um')
print()
print('Phase C Improvements:')
print('  PINN domain: 30 -> 40 um (BM2~OPD)')
print('  Design variables: 4D BM (Phase D: +4D AR = 8D)')
print('  Parametric PINN: single model for all designs')
print('  Pure PINN: no hard mask, no slit_dist')
print('  3-stage curriculum training')